\title{}
\author{}
\date{}
\makeatletter
\renewcommand{\maketitle}{}
\makeatother

\thispagestyle{empty}

\begin{center}
\vspace*{4cm}

{\LARGE Asset Allocation \& Investment Strategies \\[0.5cm]}

Academic year: 2025--2026\\[1.5cm]

Group 8\\[0.3cm]
Sacha Mimoun\\
Isabelle Chuah\\
Victor Lotigie\\
Evelyn Wang\\
Bolun Tian\\[1.5cm]

\textit{Imperial College Business School}

\end{center}

\newpage

\setcounter{secnumdepth}{0}

\thispagestyle{empty}
\clearpage

\tableofcontents

\newpage

<CENTER>
<p><font size="5"> ASSET ALLOCATION </span></p>
<p><font size="5"> ASSIGNMENT 2 : FAMA-FRENCH FACTORS (PAPER 1993) </font></p>
</p>
</CENTER>

In [54]:
import pandas as pd
import datetime
import os
import sys

In [68]:
# Import raw data

path = "raw_data/"
filename_factors = 'F-F_Research_Data_Factors.CSV'
filename_portfolios = '25_Portfolios_5x5.CSV'
filepath = os.path.join(path, filename_factors)
filepath_portfolios = os.path.join(path, filename_portfolios)

In [69]:
# We have to be prudent because the file contains in reality two datasets. One with the monthly frequency, 
# and another one below with annual data. Let's divide the two:

df_temp = pd.read_csv(filepath, skiprows=3, header=None)
separator_idx = df_temp[df_temp[0].astype(str).str.contains('Annual', case=False, na=False)].index[0]

# stop before the empty line before separator
df_ff_monthly = pd.read_csv(filepath, skiprows=3, nrows=separator_idx-1)

# skip the separator line and the blank line
df_ff_annual = pd.read_csv(filepath, skiprows=3+separator_idx+2)

In [70]:
df_temp_port = pd.read_csv(filepath_portfolios, skiprows=15, header=None)
sep_idx = df_temp_port[df_temp_port[0].astype(str).str.contains('Average Equal Weighted Returns -- Monthly', case=False, na=False)].index[0]
print(sep_idx)

1171


In [66]:
"""Cleaning Phase"""

# Clean monthly data: remove rows where first column is not a valid date (6 digits)
# First convert to string and strip whitespace, then check if it's exactly 6 digits
df_ff_monthly['date_str'] = df_ff_monthly.iloc[:, 0].astype(str).str.strip()
df_ff_monthly = df_ff_monthly[df_ff_monthly['date_str'].str.match(r'^\d{6}$')]
df_ff_monthly.index = pd.to_datetime(df_ff_monthly['date_str'], format='%Y%m').dt.strftime('%Y-%m-%d')
df_ff_monthly = df_ff_monthly.iloc[:, 1:-1]  # Drop first column and the temporary date_str column

# Clean annual data: remove rows where first column is not a valid 4-digit year
df_ff_annual['year_str'] = df_ff_annual.iloc[:, 0].astype(str).str.strip()
df_ff_annual = df_ff_annual[df_ff_annual['year_str'].str.match(r'^\d{4}$')]
df_ff_annual.index = df_ff_annual['year_str']
df_ff_annual = df_ff_annual.iloc[:, 1:-1]  # Drop first column and the temporary year_str column

print("Monthly Data:")
print(df_ff_monthly.head())
print(f"\nMonthly shape: {df_ff_monthly.shape}")
print("\nAnnual Data:")
print(df_ff_annual.head())
print(f"\nAnnual shape: {df_ff_annual.shape}")

Monthly Data:
            Mkt-RF   SMB   HML    RF
date_str                            
1926-07-01    2.96 -2.56 -2.43  0.22
1926-08-01    2.64 -1.17  3.82  0.25
1926-09-01    0.36 -1.40  0.13  0.23
1926-10-01   -3.24 -0.09  0.70  0.32
1926-11-01    2.53 -0.10 -0.51  0.31

Monthly shape: (1170, 4)

Annual Data:
          Mkt-RF    SMB    HML    RF
year_str                            
1927       29.47  -2.04  -4.54  3.12
1928       35.39   4.51  -6.17  3.56
1929      -19.54 -30.70  11.67  4.75
1930      -31.23  -5.17 -11.54  2.41
1931      -45.11   3.70 -13.95  1.07

Annual shape: (97, 4)


The dataframes shape are consistent because 97 years is equivalent to 97*12 = 1164 but one must add the 6 last months of 1926 : 1170.

Let's do the same for the 25 portfolios